# core.cache
> Lightweight, pluggable cache for CogitareLink.
Features
--------
* `memoize()` decorator         → drop-in replacement for `lru_cache`.
* Explicit `get/set/delete`     → handle unhashable keys or external sources.
* `stats()`                     → expose hit/miss/size for debugging.
* Subclass `BaseCache` to plug a different backend (e.g. diskcache).

In [ ]:
#| default_exp core.cache

In [ ]:
#| export
from __future__ import annotations
import functools, time
from typing import Any, Callable, Dict, Tuple
from dataclasses import dataclass, field
from cogitarelink.core.debug import get_logger


In [ ]:
#| export
@dataclass
class CacheStats:
    hits: int = 0
    misses: int = 0
    sets: int = 0
    def as_dict(self): return self.__dict__

## Base Cache

In [ ]:
#| export
class BaseCache:
    """Interface backbone—override storage primitives to change backend."""
    def __init__(self,maxsize:int=1024,ttl:float|None=None):
        self.maxsize,self.ttl = maxsize,ttl
        self.stats = CacheStats()
        self.log = get_logger("cache")
    def _store(self,key:str,val:Any): ...
    def _load(self,key:str) -> Any|None: ...
    def _evict(self): ...
    def set(self,key:str,val:Any):
        self._store(key,(val,time.time())); self.stats.sets += 1
    def get(self,key:str) -> Any|None:
        item = self._load(key)
        if item is None: self.stats.misses += 1; return None
        val,ts = item
        if self.ttl and time.time() - ts > self.ttl:
            self.delete(key); self.stats.misses += 1; return None
        self.stats.hits += 1
        return val
    def delete(self,key:str): ...
    def clear(self): ...
    # ---------------------------------------------------------------------   
    #  Memoization helper
    # ---------------------------------------------------------------------
    def memoize(self, ns: str = "default", maxsize: int | None = None):
        """
        Return a decorator that wraps *fn* with an LRU cache that is **scoped
        to a namespace**.

        Args:
            ns:   Arbitrary namespace label (e.g. "http", "context", "expand").
            maxsize: Optional per-namespace cache size.  Falls back to the
                     cache-wide `self.maxsize` default.

        Example
        -------
        >>> cache = InMemoryCache(maxsize=16)
        >>> @cache.memoize("math")              # math namespace
        ... def add(a, b): return a + b
        >>> add(1, 2); add(1, 2)                # 2nd hit is cached
        3
        """
        if not hasattr(self, "_memo_tables"):
            self._memo_tables: dict[str, Callable] = {}          # type: ignore

        if ns not in self._memo_tables:
            # one independent functools.lru_cache per namespace
            self._memo_tables[ns] = functools.lru_cache(
                maxsize or self.maxsize
            )

        def decorator(fn: Callable):
            wrapped = self._memo_tables[ns](fn)
            functools.update_wrapper(wrapped, fn)
            return wrapped

        return decorator
    
    def info(self): return self.stats.as_dict()

In [ ]:
#| export
class InMemoryCache(BaseCache):
    "Simple LRU dict backend."
    def __init__(self, maxsize:int=1024, ttl:float|None=None):
        super().__init__(maxsize, ttl)
        from collections import OrderedDict
        self._d: "OrderedDict[str,Tuple[Any,float]]" = OrderedDict()
    # storage ops
    def _store(self, key, val):
        d = self._d
        if key in d: d.pop(key)
        d[key] = val
        if len(d) > self.maxsize: d.popitem(last=False)  # FIFO eviction
    def _load(self, key): return self._d.get(key)
    def _evict(self): self._d.popitem(last=False)
    def delete(self, key): self._d.pop(key, None)
    def clear(self): self._d.clear()

In [ ]:
#| export
# alias for default use
Cache = InMemoryCache

In [ ]:
#| hide
from fastcore.test import *

c = Cache(maxsize=2, ttl=0.5)
c.set("a", 1)
test_eq(c.get("a"), 1)
test_eq(c.info()["hits"], 1)
# eviction
c.set("b", 2); c.set("c", 3)
test_eq(c.get("a"), None)
# ttl expiry
time.sleep(0.6)
test_eq(c.get("b"), None)
test_eq(c.info()["misses"] >= 1, True)

In [ ]:
#| hide
from fastcore.test import test_eq, test_ne

calls = {"sq": 0, "cube": 0}
cache = InMemoryCache(maxsize=8)

@cache.memoize("square")
def square(x):
    calls["sq"] += 1
    return x * x

@cache.memoize("cube")
def cube(x):
    calls["cube"] += 1
    return x * x * x

# identical args, first call populates cache
test_eq(square(3), 9)
test_eq(square(3), 9)
test_eq(calls["sq"], 1)          # only executed once

# different namespace → separate cache
test_eq(cube(3), 27)
test_eq(cube(3), 27)
test_eq(calls["cube"], 1)

# confirm independent memo tables
test_ne(cache._memo_tables["square"], cache._memo_tables["cube"])

## Diskcache

In [ ]:
#| export
class DiskCache(BaseCache):
    def __init__(self, directory=".cog_cache", **kw):
        super().__init__(**kw)
        from diskcache import Cache as DC
        self._dc = DC(str(directory))
        self._keys = []
    def _k(self, k): return str(k)
    def _store(self, k, v):
        k2 = self._k(k)
        if k2 in self._keys: self._keys.remove(k2)
        self._keys.append(k2)
        self._dc.set(k2, v)
        if len(self._keys) > self.maxsize:
            rm = self._keys.pop(0); self._dc.pop(rm, None)
    def _load(self, k): return self._dc.get(self._k(k), default=None)
    def delete(self, k):
        k2 = self._k(k); self._dc.pop(k2, None)
        if k2 in self._keys: self._keys.remove(k2)
    def clear(self):
        self._dc.clear(); self._keys = []

In [ ]:
#| hide
from fastcore.test import test_eq, test
import operator, shutil, tempfile, time
from pathlib import Path
try: from cogitarelink.core.cache import DiskCache
except ImportError: print("diskcache not present; skipping DiskCache tests")
else:
    tmp=Path(tempfile.mkdtemp())
    try:
        dc=DiskCache(directory=tmp, maxsize=3, ttl=0.5)
        dc.set("x",42)
        test_eq(dc.get("x"),42)
        dc.set("y",1); dc.set("z",2); dc.set("w",3)
        test_eq(dc.get("x"),None)
        time.sleep(0.6)
        test_eq(dc.get("y"),None)
        s=dc.info()
        test(s["hits"],0,operator.gt)
        test(s["misses"],1,operator.gt)
    finally: dc.clear(); shutil.rmtree(tmp,ignore_errors=True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()